In [1]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")

files = list(RAW_DIR.glob("*.ods")) #looks for files 

# Checks and prints out existing files 
files = sorted(RAW_DIR.glob("*.ods"))

print("ODS files found:")
for i, file in enumerate(files):
    print(f"{i}: {file}")

if not files:
    raise FileNotFoundError(f"No .ods files found in {RAW_DIR}")

# Selects first raw file, can be changed later
file_path = files[3]

print()
print("Selected file:")
print(file_path)
print("Suffix:", file_path.suffix)

ods_file = pd.ExcelFile(file_path, engine="odf")


ODS files found:
0: ../data/raw/neuzulassung_test_jan_march.ods
1: ../data/raw/neuzulassungen_2024.ods
2: ../data/raw/neuzulassungen_2025.ods
3: ../data/raw/neuzulassungen_pkw_2024_halbjahr_1.ods

Selected file:
../data/raw/neuzulassungen_pkw_2024_halbjahr_1.ods
Suffix: .ods


In [26]:


# Inspect Sheet names 
print("Sheet names:")
for i, sheet in enumerate(ods_file.sheet_names):
    print(f"{i}: {sheet}")

Sheet names:
0: Tabelle1
1: Tabelle2
2: Tabelle3


In [27]:
# Selects the sheet with the table you want to work with
sheet_name = ods_file.sheet_names[0] 

df = pd.read_excel(
    file_path,
    sheet_name=sheet_name,
    header=None,
    engine="odf"
)

print("Selected sheet:", sheet_name)
print("Shape:", df.shape)

# This automatically provides a new 0-based index.   

display(df.head(10))




Selected sheet: Tabelle1
Shape: (28, 16)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,Kfz-Neuzulassungen nach Kfz-Art und Kraftstoff...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Kfz-Art,1. Halbjahr 2020,NaN,NaN,1. Halbjahr 2021,NaN,NaN,1. Halbjahr 2022,NaN,NaN,1. Halbjahr 2023,NaN,NaN,1. Halbjahr 2024,NaN,NaN
2,NaN,absolut,Anteil in %,Veränderung in %,absolut,Anteil in %,Veränderung in %,absolut,Anteil in %,Veränderung in %,absolut,Anteil in %,Veränderung in %,absolut,Anteil in %,Veränderung in %
3,Kfz insgesamt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Personenkraftwagen,112787,69.256512,-35.883326,134396,64.356346,19.159123,108606,67.905488,-19.189559,126690,69.157705,16.651014,135113,68.552569,6.648512
5,Lkw N1,16818,10.327041,-27.439814,30783,14.740628,83.036033,11252,7.03527,-63.447357,14538,7.936023,29.203697,19560,9.924199,34.543954
6,Lkw N2 und N3,1774,1.089319,-43.521172,2133,1.0214,20.236753,1626,1.01665,-23.769339,1926,1.051367,18.450185,2586,1.312064,34.267913
7,Motorräder,16451,10.101686,-7.950985,20618,9.873055,25.329767,20705,12.945722,0.421961,22594,12.333643,9.1234,22119,11.222564,-2.102328
8,Motorfahrräder,6391,3.924374,-8.188479,7278,3.485115,13.878892,5978,3.737722,-17.86205,5973,3.260549,-0.08364,6219,3.155347,4.118533
9,Andere Kfz,8633,5.301067,-16.806399,13623,6.523457,57.80146,11770,7.359148,-13.601997,11469,6.260713,-2.557349,11497,5.833257,0.244136


In [ ]:
bundeslaender = [
    "Österreich",
    "Burgenland",
    "Kärnten",
    "Niederösterreich",
    "Oberösterreich",
    "Salzburg",
    "Steiermark",
    "Tirol",
    "Vorarlberg",
    "Wien",
]

df = pd.read_excel(
    file_path,
    sheet_name=sheet_name,
    header=None,
    engine="odf"
)

# rename first column
df = df.rename(columns={df.columns[0]: "Bezeichnung"})
df["Bezeichnung"] = df["Bezeichnung"].astype(str).str.strip()

# Rename Kraftstoff columns
fuel_cols = df.iloc[1, 1:].tolist()
df.columns = ["Bezeichnung"] + fuel_cols


# detect Bundesland rows
df["Bundesland"] = df["Bezeichnung"].where(df["Bezeichnung"].isin(bundeslaender))
df["Bundesland"] = df["Bundesland"].ffill()

# remove rows before the first Bundesland section
df = df[df["Bundesland"].notna()].copy()

# remove last row (Fußnote)
df = df.iloc[:-1].copy()

# create Fahrzeugklasse column
df["Fahrzeugklasse"] = df["Bezeichnung"]

# Bundesland section rows represent totals
df.loc[df["Bezeichnung"].isin(bundeslaender), "Fahrzeugklasse"] = "All"

# replace obvious missing values
df = df.replace("-", pd.NA)


# identify value columns
value_cols = [
    c for c in df.columns
    if c not in ["Bundesland", "Fahrzeugklasse", "Bezeichnung"]
]

# convert value columns to numeric
for col in value_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace(["-", "nan", ""], pd.NA)
    )

    df[col] = pd.to_numeric(df[col], errors="coerce")

# reorder columns
df = df[
    ["Bundesland", "Fahrzeugklasse"] + value_cols
]    

#Clean column names, no trailing spaces
df.columns = df.columns.str.strip()


display(df.head(5))

,Bundesland,Fahrzeugklasse,Benzin,Diesel,Elektro,Flüssiggas,Erdgas,Benzin/Flüssiggas (bivalent),Benzin/Erdgas (bivalent),Benzin/Elektro (hybrid),Diesel/Elektro (hybrid),Wasserstoff(Brennstoffzelle)
2,Österreich,All,143457.0,92752.0,51670.0,1.0,45.0,5.0,2.0,66811.0,14500.0,3.0
3,Österreich,Personenkraftwagen Klasse M1,84004.0,44132.0,44622.0,NaN,11.0,1.0,NaN,66672.0,14346.0,1.0
4,Österreich,Motorräder Klasse L3e,45197.0,NaN,1303.0,NaN,NaN,NaN,NaN,8.0,NaN,NaN
5,Österreich,Motorfahrräder Klasse L1e,9843.0,NaN,1655.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Österreich,Vierrädrige Kraftfahrzeuge Klasse L7e,35.0,NaN,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# Checkpoint, Bundesländer und Fahrzeugtyp
print(df["Bundesland"].value_counts(dropna=False))

print(df["Fahrzeugklasse"].unique())

display(df.tail(10))


Bundesland
Österreich          19
Burgenland          19
Kärnten             19
Niederösterreich    19
Oberösterreich      19
Salzburg            19
Steiermark          19
Tirol               19
Vorarlberg          19
Wien                19
Name: count, dtype: int64
<StringArray>
[                                        'All',
                'Personenkraftwagen Klasse M1',
                       'Motorräder Klasse L3e',
                   'Motorfahrräder Klasse L1e',
       'Vierrädrige Kraftfahrzeuge Klasse L7e',
                   'Motordreiräder Klasse L5e',
      'Dreirädrige Kleinkrafträder Klasse L2e',
 'Vierrädrige Leichtkraftfahrzeuge Klasse L6e',
                  'Omnibusse Klasse M2 und M3',
                    'Lastkraftwagen Klasse N1',
                    'Lastkraftwagen Klasse N2',
                    'Lastkraftwagen Klasse N3',
 'Land- und forstwirtschaftliche Zugmaschinen',
                          'Sattelzugfahrzeuge',
                  'Motor- und Transportkarren',

,Bundesland,Fahrzeugklasse,Benzin,Diesel,Elektro,Flüssiggas,Erdgas,Benzin/Flüssiggas (bivalent),Benzin/Erdgas (bivalent),Benzin/Elektro (hybrid),Diesel/Elektro (hybrid),Wasserstoff(Brennstoffzelle)
182,Wien,Lastkraftwagen Klasse N1,249.0,5202.0,1566.0,NaN,NaN,NaN,NaN,10.0,8.0,1.0
183,Wien,Lastkraftwagen Klasse N2,NaN,140.0,23.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
184,Wien,Lastkraftwagen Klasse N3,NaN,273.0,22.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
185,Wien,Land- und forstwirtschaftliche Zugmaschinen,103.0,56.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
186,Wien,Sattelzugfahrzeuge,NaN,80.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187,Wien,Motor- und Transportkarren,NaN,36.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188,Wien,Selbstfahrende Arbeitsmaschinen,NaN,312.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
189,Wien,Erntemaschinen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190,Wien,Wohnmobile,NaN,99.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
191,Wien,Sonstige Kraftfahrzeuge,NaN,98.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "ev_registrations_2024.csv"

df.to_csv(output_path, index=False)

df_check = pd.read_csv(output_path)

print(f"Saved processed file to: {output_path}")
print("File exists:", output_path.exists())
print()

# Prüfen ob Dateipfad existiert 
print(output_path)
print(output_path.exists())
print()

# Last control
display(df_check.head()) # Spaltennamen ?
print(df_check.shape) # Alle Zeilen/Spalten vorhanden ?
print(df_check.dtypes) # Richtiger datatype ?

Saved processed file to: ../data/processed/ev_registrations_2024.csv
File exists: True

../data/processed/ev_registrations_2024.csv
True



,Bundesland,Fahrzeugklasse,Benzin,Diesel,Elektro,Flüssiggas,Erdgas,Benzin/Flüssiggas (bivalent),Benzin/Erdgas (bivalent),Benzin/Elektro (hybrid),Diesel/Elektro (hybrid),Wasserstoff(Brennstoffzelle)
0,Österreich,All,143457.0,92752.0,51670.0,1.0,45.0,5.0,2.0,66811.0,14500.0,3.0
1,Österreich,Personenkraftwagen Klasse M1,84004.0,44132.0,44622.0,NaN,11.0,1.0,NaN,66672.0,14346.0,1.0
2,Österreich,Motorräder Klasse L3e,45197.0,NaN,1303.0,NaN,NaN,NaN,NaN,8.0,NaN,NaN
3,Österreich,Motorfahrräder Klasse L1e,9843.0,NaN,1655.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Österreich,Vierrädrige Kraftfahrzeuge Klasse L7e,35.0,NaN,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


(190, 12)
Bundesland                          str
Fahrzeugklasse                      str
Benzin                          float64
Diesel                          float64
Elektro                         float64
Flüssiggas                      float64
Erdgas                          float64
Benzin/Flüssiggas (bivalent)    float64
Benzin/Erdgas (bivalent)        float64
Benzin/Elektro (hybrid)         float64
Diesel/Elektro (hybrid)         float64
Wasserstoff(Brennstoffzelle)    float64
dtype: object


In [ ]:
# Inspect all sheets 
for sheet in ods_file.sheet_names:
    print("\n" + "=" * 80)
    print(f"Sheet: {sheet}")
    print("=" * 80)

    df_sheet = pd.read_excel(
        file_path,
        sheet_name=sheet,
        header=None,
        engine="odf"
    )

    print("Shape:", df_sheet.shape)
    display(df_sheet.head(20))


Sheet: Jänner
Shape: (193, 8)


,0,1,2,3,4,5,6,7
0,Kfz-Neuzulassungen nach Bundesland und Kraftst...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bundesland / Kraftfahrzeug,Benzin,Diesel,Elektro,Erdgas,Benzin/Flüssiggas (bivalent),Benzin/Elektro (hybrid),Diesel/Elektro (hybrid)
2,Österreich,7355,5348,5209,1,1,8353,969
3,Personenkraftwagen Klasse M1,6718,2275,4702,-,-,8275,959
4,Motorräder Klasse L3e,281,-,42,-,-,-,-
5,Motorfahrräder Klasse L1e,127,-,33,-,-,-,-
6,Vierrädrige Kraftfahrzeuge Klasse L7e,-,-,9,-,-,-,-
7,Motordreiräder Klasse L5e,3,-,-,-,-,-,-
8,Dreirädrige Kleinkrafträder Klasse L2e,-,-,10,-,-,-,-
9,Vierrädrige Leichtkraftfahrzeuge Klasse L6e,-,31,25,-,-,-,-



Sheet: Februar
Shape: (193, 8)


,0,1,2,3,4,5,6,7
0,Kfz-Neuzulassungen nach Bundesland und Kraftst...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bundesland / Kraftfahrzeug,Benzin,Diesel,Elektro,Erdgas,Benzin/Flüssiggas (bivalent),Benzin/Elektro (hybrid),Diesel/Elektro (hybrid)
2,Österreich,7261,6432,5110,-,-,7982,936
3,Personenkraftwagen Klasse M1,5530,2503,4439,-,-,7903,913
4,Motorräder Klasse L3e,1071,-,85,-,-,-,-
5,Motorfahrräder Klasse L1e,407,-,44,-,-,-,-
6,Vierrädrige Kraftfahrzeuge Klasse L7e,1,-,8,-,-,-,-
7,Motordreiräder Klasse L5e,8,-,-,-,-,-,-
8,Dreirädrige Kleinkrafträder Klasse L2e,-,-,25,-,-,-,-
9,Vierrädrige Leichtkraftfahrzeuge Klasse L6e,-,38,40,-,-,-,-



Sheet: März
Shape: (193, 9)


,0,1,2,3,4,5,6,7,8
0,Kfz-Neuzulassungen nach Bundesland und Kraftst...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bundesland / Kraftfahrzeug,Benzin,Diesel,Elektro,Erdgas,Benzin/Flüssiggas (bivalent),Benzin/Erdgas (bivalent),Benzin/Elektro (hybrid),Diesel/Elektro (hybrid)
2,Österreich,14842,7654,9233,-,1,1,12395,1426
3,Personenkraftwagen Klasse M1,8181,2953,8206,-,-,-,12282,1396
4,Motorräder Klasse L3e,4882,-,239,-,-,-,-,-
5,Motorfahrräder Klasse L1e,1348,-,132,-,-,-,-,-
6,Vierrädrige Kraftfahrzeuge Klasse L7e,-,-,9,-,-,-,-,-
7,Motordreiräder Klasse L5e,20,-,-,-,-,-,-,-
8,Dreirädrige Kleinkrafträder Klasse L2e,-,-,62,-,-,-,-,-
9,Vierrädrige Leichtkraftfahrzeuge Klasse L6e,-,57,63,-,-,-,-,-



Sheet: Jänner-März
Shape: (193, 9)


,0,1,2,3,4,5,6,7,8
0,Kfz-Neuzulassungen nach Bundesland und Kraftst...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Bundesland / Kraftfahrzeug,Benzin,Diesel,Elektro,Erdgas,Benzin/Flüssiggas (bivalent),Benzin/Erdgas (bivalent),Benzin/Elektro (hybrid),Diesel/Elektro (hybrid)
2,Österreich,29458,19434,19552,1,2,1,28730,3331
3,Personenkraftwagen Klasse M1,20429,7731,17347,-,-,-,28460,3268
4,Motorräder Klasse L3e,6234,-,366,-,-,-,-,-
5,Motorfahrräder Klasse L1e,1882,-,209,-,-,-,-,-
6,Vierrädrige Kraftfahrzeuge Klasse L7e,1,-,26,-,-,-,-,-
7,Motordreiräder Klasse L5e,31,-,-,-,-,-,-,-
8,Dreirädrige Kleinkrafträder Klasse L2e,-,-,97,-,-,-,-,-
9,Vierrädrige Leichtkraftfahrzeuge Klasse L6e,-,126,128,-,-,-,-,-
